# 1+2

This notebook merges the two DFs from A1 and A2. A .fasta will be created as well, which will be the foundation for the DB used in the pipeline.

In [1]:
import pandas as pd
import requests
from Bio import SeqIO
from io import StringIO

In [2]:
df1 = pd.read_csv("../Approach 1/a1_df.tsv", sep="\t")
df2 = pd.read_csv("../Approach 2/transporters2_df.tsv", sep="\t")

In [3]:
df = pd.concat([df1, df2], ignore_index=True)
df

,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
2,A0CS82,9.B.82.1.5,NaN,NaN,MIIEEQIEEKMIYKAIHRVKVNYQKKIDRYILYKKSRWFFNLLLML...,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),MSQPITYSSLISLSLAKFPQVYMYTDGFMSNDFELISFNSVHGNLF...,1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,MINKAGKQLTWLFALKILSRIFDLSLNILVLRDLEPGIYGLTTNLD...,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80117,P26905,3.A.1.5.2,CHEBI:17627,ferroheme b,MEKVLSVQNLHVSFTTYGGTVQAVRGVSFDLYKGETFAIVGESGCG...,3.A.1,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,ferroheme b (out) + ATP → ferroheme b (in) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
80118,P45095,3.A.1.5.27,CHEBI:16856,glutathione,MALLDVKELSVHFGDKKTPFKAVDRISYQVAQGEVLGIVGESGSGK...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,glutathione (in) + ATP → glutathione (out) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
80119,P45095,3.A.1.5.27,CHEBI:16856,glutathione,MALLDVKELSVHFGDKKTPFKAVDRISYQVAQGEVLGIVGESGSGK...,3.A.1,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,glutathione (out) + ATP → glutathione (in) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
80120,P45094,3.A.1.5.27,CHEBI:16856,glutathione,MTNEVKENTPLLNAIGLKKYYPVKKGLFAKPQQVKALDGVSFQLER...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,glutathione (in) + ATP → glutathione (out) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...


Noticing that some rows have missing TCIDs for known AAs that are present in TCDB. Therfore, a mapping of TCIDs are performed.

In [4]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def parse_data(substrates_txt, aa_txt):

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
                        for line in substrates_lines
                        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])

    # AA sequence (TCID, UID and AA)
    fasta_io = StringIO(aa_txt)
    tc_data = [[record.description.split("|")[3].split()[0],  # TCID
                record.description.split("|")[2],  # UID
                str(record.seq)]  # AA
                for record in SeqIO.parse(fasta_io, "fasta")]
    
    df_aa = pd.DataFrame(tc_data, columns=["TCID", "UID", "AA"])

    return df_substrates, df_aa

tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"
tc_aa_url = "https://www.tcdb.org/public/tcdb"

tc_substrates_txt = fetch_data(tc_substrates_url)
tc_aa_txt = fetch_data(tc_aa_url)


df_substrates, df_aa = parse_data(tc_substrates_txt, tc_aa_txt)

In [5]:
df_aa_unique = df_aa.drop_duplicates(subset="AA")
df.loc[df["TCID"].isna(), "TCID"] = df["AA"].map(df_aa_unique.set_index("AA")["TCID"])
df

,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
2,A0CS82,9.B.82.1.5,NaN,NaN,MIIEEQIEEKMIYKAIHRVKVNYQKKIDRYILYKKSRWFFNLLLML...,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),MSQPITYSSLISLSLAKFPQVYMYTDGFMSNDFELISFNSVHGNLF...,1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,MINKAGKQLTWLFALKILSRIFDLSLNILVLRDLEPGIYGLTTNLD...,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80117,P26905,3.A.1.5.2,CHEBI:17627,ferroheme b,MEKVLSVQNLHVSFTTYGGTVQAVRGVSFDLYKGETFAIVGESGCG...,3.A.1,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,ferroheme b (out) + ATP → ferroheme b (in) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
80118,P45095,3.A.1.5.27,CHEBI:16856,glutathione,MALLDVKELSVHFGDKKTPFKAVDRISYQVAQGEVLGIVGESGSGK...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,glutathione (in) + ATP → glutathione (out) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
80119,P45095,3.A.1.5.27,CHEBI:16856,glutathione,MALLDVKELSVHFGDKKTPFKAVDRISYQVAQGEVLGIVGESGSGK...,3.A.1,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,glutathione (out) + ATP → glutathione (in) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
80120,P45094,3.A.1.5.27,CHEBI:16856,glutathione,MTNEVKENTPLLNAIGLKKYYPVKKGLFAKPQQVKALDGVSFQLER...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,glutathione (in) + ATP → glutathione (out) + A...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...


Now, from these newly obtained TCIDs, one also needs to extract their ChEBI information, family, mechanism and so on. Whatever is retrievable from TCDB.

In [6]:
df_updated = df.merge(df_substrates, on="TCID", how="left", suffixes=("", "_new"))

df_updated["CHEBI ID"] = df_updated["CHEBI ID"].fillna(df_updated["CHEBI ID_new"])
df_updated["CHEBI Name"] = df_updated["CHEBI Name"].fillna(df_updated["CHEBI Name_new"])

df_updated = df_updated.drop(columns=["CHEBI ID_new", "CHEBI Name_new"])

# Need to do s2p on the column again...
df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
df_updated["CHEBI ID"] = df["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))

df_updated = df_updated.drop_duplicates()
df_updated

,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier
0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
1,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
2,A0CS82,9.B.82.1.5,NaN,NaN,MIIEEQIEEKMIYKAIHRVKVNYQKKIDRYILYKKSRWFFNLLLML...,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),MSQPITYSSLISLSLAKFPQVYMYTDGFMSNDFELISFNSVHGNLF...,1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,MINKAGKQLTWLFALKILSRIFDLSLNILVLRDLEPGIYGLTTNLD...,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
347074,O06926,3.B.1.1.4,NaN,sodium(1+),MAKWTELQDKSFLEATARERAVGIVDEGTFTEFCGPFDKIYSPHLP...,3.B.1,NaN,NaN,NaN,RHEA:23028,N(6)-biotinyl-L-lysyl-[protein] + malonyl-[ACP...,N(6)-biotinyl-L-lysine residue;O-(S-malonylpan...,CHEBI:83144;CHEBI:78449;CHEBI:83145;CHEBI:78446
347075,O06927,3.B.1.1.4,NaN,sodium(1+),MEIMMGQGRLAIEKIVDPESFKENTIGESSFEDNEVGPGAVVGTAQ...,3.B.1,NaN,NaN,NaN,RHEA:23028,N(6)-biotinyl-L-lysyl-[protein] + malonyl-[ACP...,N(6)-biotinyl-L-lysine residue;O-(S-malonylpan...,CHEBI:83144;CHEBI:78449;CHEBI:83145;CHEBI:78446
347082,P26905,3.A.1.5.2,NaN,5-aminolevulinic acid,MEKVLSVQNLHVSFTTYGGTVQAVRGVSFDLYKGETFAIVGESGCG...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,5-aminolevulinic acid (in) + ATP → 5-aminolevu...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
347085,P26905,3.A.1.5.2,NaN,5-aminolevulinic acid,MEKVLSVQNLHVSFTTYGGTVQAVRGVSFDLYKGETFAIVGESGCG...,3.A.1,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,5-aminolevulinic acid (out) + ATP → 5-aminolev...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...


The next step is to extract all families from the previous Nan-TCIDs, and then their mechanisms, acting entities and reactions.

In [7]:
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_final.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "⇌", "â†’": "→"}, regex=True)

# Creating "Family" only for Nan-values
df_updated.loc[df_updated["Family"].isna(), "Family"] = df_updated.loc[df_updated["Family"].isna(), "TCID"].apply(
    lambda x: ".".join(str(x).split(".")[:3]) if pd.notna(x) else x)

df_updated = df_updated.merge( df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left",suffixes=("", "_new"))

df_updated["Mechanism"] = df_updated["Mechanism"].fillna(df_updated["Mechanism_new"])
df_updated["Acting Entity"] = df_updated["Acting Entity"].fillna(df_updated["Acting Entity_new"])

df_updated = df_updated.drop(columns=["Mechanism_new", "Acting Entity_new"])

def create_reaction_row(row):
    if pd.notna(row["Reaction"]):
        return row["Reaction"]

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    
    mechanisms = str(row["Mechanism"]).split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = [mechanism.replace(entity, chebi_name) for mechanism, entity in zip(mechanisms, acting_entities)]
    
    return ", ".join(reactions)

df_updated["Reaction"] = df_updated.apply(create_reaction_row, axis=1)

Closing in now. The final touch is to implement the actual accession ID used by TCDB. The UID column MOSTLY correspond to the given AA, but sometimes, they have been RSIDs. UID is the final column after all RSIDs have been converted to UIDs, in order to map between TCDB-UniProt-Rhea, both ways. For referencing purposes towards the transporters, AAs and TCIDs in TCDB, a new column is added. That is the AID (Accession ID), which is the ID used by TCDB for any TCID (or a protein in a larger complex that all have the same TCID). Finally, just some minor readjustments for the columns, making it just a tad easier to look at, before the final DF is obtained.

In [8]:
df_temp = df_updated.drop_duplicates().reset_index(drop=True)
df_temp = df_temp.merge(df_aa[["AA", "UID"]], on="AA", how="left", suffixes=("", "_new"))
df_temp.rename(columns={"UID_new":"AID"}, inplace=True)
cols = ["AID"] + [col for col in df_temp.columns if col != "AID"]
df_final= df_temp[cols]
df_final

,AID,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier
0,A0CIB0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cl- (out) ⇌ Cl- (in),Cl-,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
1,A0CIB0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cations (out) ⇌ Cations (in),Cations,chloride (out) ⇌ chloride (in),NaN,NaN,NaN,NaN
2,A0CS82,A0CS82,9.B.82.1.5,NaN,NaN,MIIEEQIEEKMIYKAIHRVKVNYQKKIDRYILYKKSRWFFNLLLML...,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),MSQPITYSSLISLSLAKFPQVYMYTDGFMSNDFELISFNSVHGNLF...,1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,MINKAGKQLTWLFALKILSRIFDLSLNILVLRDLEPGIYGLTTNLD...,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85506,O06926,O06926,3.B.1.1.4,NaN,sodium(1+),MAKWTELQDKSFLEATARERAVGIVDEGTFTEFCGPFDKIYSPHLP...,3.B.1,NaN,NaN,NaN,RHEA:23028,N(6)-biotinyl-L-lysyl-[protein] + malonyl-[ACP...,N(6)-biotinyl-L-lysine residue;O-(S-malonylpan...,CHEBI:83144;CHEBI:78449;CHEBI:83145;CHEBI:78446
85507,O06927,O06927,3.B.1.1.4,NaN,sodium(1+),MEIMMGQGRLAIEKIVDPESFKENTIGESSFEDNEVGPGAVVGTAQ...,3.B.1,NaN,NaN,NaN,RHEA:23028,N(6)-biotinyl-L-lysyl-[protein] + malonyl-[ACP...,N(6)-biotinyl-L-lysine residue;O-(S-malonylpan...,CHEBI:83144;CHEBI:78449;CHEBI:83145;CHEBI:78446
85508,P26905,P26905,3.A.1.5.2,NaN,5-aminolevulinic acid,MEKVLSVQNLHVSFTTYGGTVQAVRGVSFDLYKGETFAIVGESGCG...,3.A.1,Solute (in) + ATP → Solute (out) + ADP + Pi,Solute,5-aminolevulinic acid (in) + ATP → 5-aminolevu...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...
85509,P26905,P26905,3.A.1.5.2,NaN,5-aminolevulinic acid,MEKVLSVQNLHVSFTTYGGTVQAVRGVSFDLYKGETFAIVGESGCG...,3.A.1,Substrate (out) + ATP → Substrate (in) + ADP + Pi,Substrate,5-aminolevulinic acid (out) + ATP → 5-aminolev...,RHEA:23120,a dipeptide(out) + ATP + H2O = a dipeptide(in)...,a dipeptide;ATP;H2O;ADP;phosphate;H(+),CHEBI:90799;CHEBI:30616;CHEBI:15377;CHEBI:4562...


The last step is to write the DF to a .tsv and a .fasta. The subset of identical rows for AID, TCID and AA is quite substantial, so the final .fasta is not very large (approx. 24200 transporters). But these will further be mapped to several reactions, substrates and mechanisms further down the pipeline that will come. This reduction of the subset is done to reduce BLAST-time.

In [9]:
df_fasta = df_final.drop_duplicates(subset=["AID", "TCID", "AA"])
fasta_file = "transporters.fasta"

with open(fasta_file, "w") as f:
    for _, row in df_fasta.iterrows():
        aid = row["AID"]
        tcid = row["TCID"]
        sequence = row["AA"]
        f.write(f">{aid}|{tcid}\n{sequence}\n")

df.to_csv("transporters_df.tsv", sep="\t", index=False)